# DATA FETCHING

# Cell 1: setup and load cached API data

In [3]:
import sys, json
from pathlib import Path
sys.path.append("..")

import config
from api.cricbuzz_client import CricbuzzClient
from utils import db_connection as db

client = CricbuzzClient()

live_file = Path("../data/live_raw.json")
if live_file.exists():
    live = json.loads(live_file.read_text(encoding="utf-8"))
    print("live: from data/live_raw.json")
else:
    live = client.get(config.ENDPOINTS["live_matches"])   # costs 1 call
    print("live: fetched from API, quota left:", client.quota_remaining)

cache_hit = client._cache_file(config.ENDPOINTS["recent_matches"], None).exists()
recent = client.get(config.ENDPOINTS["recent_matches"])
print("recent:", "from cache" if cache_hit else f"fetched, quota left: {client.quota_remaining}")

live: from data/live_raw.json
recent: from cache


# Cell 2: check the JSON key names

In [4]:
first = recent["typeMatches"][0]["seriesMatches"]
wrapper = next(b["seriesAdWrapper"] for b in first if "seriesAdWrapper" in b)
info = wrapper["matches"][0]["matchInfo"]

print("matchInfo keys :", sorted(info.keys()))
print("team1 keys     :", sorted(info["team1"].keys()))
print("venueInfo keys :", sorted(info.get("venueInfo", {}).keys()))
print("wrapper keys   :", sorted(wrapper.keys()))

matchInfo keys : ['endDate', 'hideScoreStatus', 'isFantasyEnabled', 'isTimeAnnounced', 'matchDesc', 'matchFormat', 'matchId', 'seriesEndDt', 'seriesId', 'seriesName', 'seriesStartDt', 'startDate', 'state', 'stateTitle', 'status', 'team1', 'team2', 'venueInfo']
team1 keys     : ['imageId', 'teamId', 'teamName', 'teamSName']
venueInfo keys : ['city', 'ground', 'id', 'latitude', 'longitude', 'timezone']
wrapper keys   : ['matches', 'seriesId', 'seriesName']


# Cell 3: flatten

In [5]:
def flatten_matches(data):
    """Flatten a Cricbuzz response into team, venue, series, and match rows."""
    tables = {"teams": [], "venues": [], "series": [], "matches": []}
    skipped = 0

    for match_type in data.get("typeMatches", []):
        for item in match_type.get("seriesMatches", []):
            wrapper = item.get("seriesAdWrapper")
            if not isinstance(wrapper, dict):
                continue

            for match in wrapper.get("matches", []):
                info = match.get("matchInfo")
                if not isinstance(info, dict) or info.get("matchId") is None:
                    skipped += 1
                    continue

                team_ids = []
                for key in ("team1", "team2"):
                    team = info.get(key)
                    if isinstance(team, dict) and team.get("teamId") is not None:
                        team_id = team["teamId"]
                        team_ids.append(team_id)
                        tables["teams"].append({
                            "team_id": team_id,
                            "team_name": team.get("teamName"),
                            "team_short_name": team.get("teamSName"),
                        })

                venue_info = info.get("venueInfo")
                venue_id = None
                if isinstance(venue_info, dict) and venue_info.get("id") is not None:
                    venue_id = venue_info["id"]
                    tables["venues"].append({
                        "venue_id": venue_id,
                        "ground": venue_info.get("ground"),
                        "city": venue_info.get("city"),
                        "timezone": venue_info.get("timezone"),
                    })

                series_id = info.get("seriesId", wrapper.get("seriesId"))
                if series_id is not None:
                    tables["series"].append({
                        "series_id": series_id,
                        "series_name": info.get("seriesName", wrapper.get("seriesName")),
                    })

                tables["matches"].append({
                    "match_id": info["matchId"],
                    "series_id": series_id,
                    "match_desc": info.get("matchDesc"),
                    "match_format": info.get("matchFormat"),
                    "start_date": info.get("startDate"),
                    "end_date": info.get("endDate"),
                    "state": info.get("state"),
                    "status": info.get("status"),
                    "team1_id": team_ids[0] if len(team_ids) > 0 else None,
                    "team2_id": team_ids[1] if len(team_ids) > 1 else None,
                    "venue_id": venue_id,
                })

    return {table: list({row[key]: row for row in rows}.values())
            for table, rows in tables.items()
            for key in ({"teams": "team_id", "venues": "venue_id",
                         "series": "series_id", "matches": "match_id"}[table],)} | {
        "skipped": skipped
    }


def merge(*results):
    """Combine flatten_matches() results, de-duplicating by primary key."""
    keys = {"teams": "team_id", "venues": "venue_id",
            "series": "series_id", "matches": "match_id"}
    out = {k: {} for k in keys}
    skipped = 0
    for res in results:
        skipped += res["skipped"]
        for table, pk in keys.items():
            for row in res[table]:
                out[table][row[pk]] = row
    return {t: list(rows.values()) for t, rows in out.items()}, skipped

rows, skipped = merge(flatten_matches(live), flatten_matches(recent))
print({t: len(r) for t, r in rows.items()}, "skipped:", skipped)

{'teams': 113, 'venues': 55, 'series': 19, 'matches': 98} skipped: 0


# Cell 4: load into the database

In [6]:
import importlib
from services import match_service
importlib.reload(match_service)          # pick up the fixed file
from services.match_service import flatten_matches

# rebuild rows from the already-loaded live/recent JSON (no API calls)
rows, skipped = merge(flatten_matches(live), flatten_matches(recent))
print({t: len(r) for t, r in rows.items()}, "skipped:", skipped)

# safety check: these must match the schema columns
print("team keys:", sorted(rows["teams"][0].keys()))
assert "short_name" in rows["teams"][0], "still the old parser, restart the kernel"
assert "team_short_name" not in rows["teams"][0]

from sqlalchemy.exc import IntegrityError
try:
    db.upsert_rows("teams", rows["teams"], ["team_id"])
    db.upsert_rows("venues", rows["venues"], ["venue_id"])
    db.upsert_rows("series", rows["series"], ["series_id"])
    db.upsert_rows("matches", rows["matches"], ["match_id"])
except IntegrityError as e:
    print("Load failed:", str(e.orig))

db.table_counts()

{'teams': 113, 'venues': 55, 'series': 19, 'matches': 98} skipped: 0
team keys: ['country', 'short_name', 'team_id', 'team_name']


{'batting_performances': 0,
 'bowling_performances': 0,
 'matches': 98,
 'player_career_stats': 0,
 'players': 0,
 'series': 19,
 'teams': 113,
 'venues': 55}

# Cell 5: check how well the status parsing worked

In [8]:
df = db.run_query("""
    SELECT m.match_id, t1.team_name AS team1, t2.team_name AS team2,
           m.state, m.winner_id, m.victory_margin, m.victory_type,
           m.toss_decision
    FROM matches m
    JOIN teams t1 ON t1.team_id = m.team1_id
    JOIN teams t2 ON t2.team_id = m.team2_id
    ORDER BY m.match_date DESC
""")
print(len(df), "API matches")
print("Completed but no winner parsed:",
      len(df[(df.state == "Complete") & df.winner_id.isna()]))
df.head(15)

98 API matches
Completed but no winner parsed: 19


,match_id,team1,team2,state,winner_id,victory_margin,victory_type,toss_decision
0,176441,Romania Women,Sweden Women,Complete,1048.0,10.0,wickets,None
1,176430,Romania Women,Austria Women,Complete,775.0,7.0,wickets,None
2,173397,Boland,Mpumalanga Rhinos,Complete,1759.0,39.0,runs,None
3,172677,Zimbabwe,Bangladesh,Complete,12.0,99.0,runs,None
4,169950,Canada,Nepal,Complete,26.0,24.0,runs,None
5,168050,Hobart Hurricanes Women,Melbourne Renegades Women,Complete,358.0,8.0,wickets,None
6,168040,Adelaide Strikers Women,Sydney Sixers Women,Complete,355.0,38.0,runs,None
7,168031,ACT Women,Melbourne Stars Women,Complete,359.0,4.0,wickets,None
8,167103,South Australia,Queensland,Complete,158.0,126.0,runs,None
9,151602,India,West Indies,Complete,NaN,NaN,no result,None


# Cell 6: run the seed

In [12]:
import importlib
from utils import seed_data
importlib.reload(seed_data)

print("has run_seed:", hasattr(seed_data, "run_seed"))

summary = seed_data.run_seed()
print(summary)
db.table_counts()

has run_seed: True
{'players': 120, 'career_rows': 277, 'series': 21, 'matches': 450, 'batting_rows': 11012, 'bowling_rows': 5350}


{'batting_performances': 11012,
 'bowling_performances': 5350,
 'matches': 548,
 'player_career_stats': 277,
 'players': 120,
 'series': 40,
 'teams': 115,
 'venues': 67}

# Cell 7: prove every query has data to work on

In [13]:
checks = {
 "Q2  matches in last 30 days":
   "SELECT COUNT(*) n FROM matches WHERE match_date >= date('now','-30 day')",
 "Q4  venues > 50,000":
   "SELECT COUNT(*) n FROM venues WHERE capacity > 50000",
 "Q8  series started in 2024":
   "SELECT COUNT(*) n FROM series WHERE start_date LIKE '2024%'",
 "Q9  all-rounders >1000 runs & >50 wkts":
   "SELECT COUNT(*) n FROM player_career_stats c JOIN players p USING(player_id) "
   "WHERE p.playing_role='All-rounder' AND c.runs>1000 AND c.wickets>50",
 "Q12 home wins":
   "SELECT COUNT(*) n FROM matches m JOIN venues v USING(venue_id) "
   "JOIN teams t ON t.team_id=m.winner_id WHERE v.country=t.country",
 "Q13 partnerships >= 100":
   "SELECT COUNT(*) n FROM batting_performances a JOIN batting_performances b "
   "ON a.match_id=b.match_id AND a.innings_no=b.innings_no "
   "AND b.batting_position=a.batting_position+1 WHERE a.runs+b.runs>=100",
 "Q14 bowler-venue combos (>=3 matches)":
   "SELECT COUNT(*) n FROM (SELECT b.player_id, m.venue_id FROM bowling_performances b "
   "JOIN matches m USING(match_id) WHERE b.overs>=4 GROUP BY 1,2 "
   "HAVING COUNT(DISTINCT b.match_id)>=3)",
 "Q22 team pairs >=5 matches (3 yrs)":
   "SELECT COUNT(*) n FROM (SELECT MIN(team1_id,team2_id), MAX(team1_id,team2_id) "
   "FROM matches WHERE match_date>=date('now','-3 year') GROUP BY 1,2 HAVING COUNT(*)>=5)",
 "Q24 player pairs >=5 partnerships":
   "SELECT COUNT(*) n FROM (SELECT a.player_id, b.player_id FROM batting_performances a "
   "JOIN batting_performances b ON a.match_id=b.match_id AND a.innings_no=b.innings_no "
   "AND b.batting_position=a.batting_position+1 GROUP BY 1,2 HAVING COUNT(*)>=5)",
}
for label, sql in checks.items():
    n = int(db.run_query(sql)["n"].iloc[0])
    print(f"{'OK  ' if n else 'EMPTY'} {label:42} {n}")

print("\nForeign-key problems:")
print(db.run_query("PRAGMA foreign_key_check"))

OK   Q2  matches in last 30 days                114
OK   Q4  venues > 50,000                        3
OK   Q8  series started in 2024                 4
OK   Q9  all-rounders >1000 runs & >50 wkts     41
OK   Q12 home wins                              171
OK   Q13 partnerships >= 100                    968
OK   Q14 bowler-venue combos (>=3 matches)      590
OK   Q22 team pairs >=5 matches (3 yrs)         28
OK   Q24 player pairs >=5 partnerships          80

Foreign-key problems:
Empty DataFrame
Columns: [table, rowid, parent, fkid]
Index: []
